In [8]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from raptor import Tensor


In [9]:
import numpy as np
from raptor import Tensor
from raptor import nn
from raptor.optim import SGD

np.random.seed(42)

x_data= np.random.randn(100,1).astype(np.float32)# 100 samples(batch size), 1 feature each
y_data = 2.0*x_data +1.0

In [10]:
model = nn.Linear(1,1)
criterion = nn.MSELoss()
optimizer = SGD(model.parameters(), lr = 0.1)

In [11]:
for epoch in range(100):
    x = Tensor(x_data, requires_grad = False)
    y = Tensor(y_data, requires_grad=False)

    preds = model(x)
    loss = criterion(preds,y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


    if epoch %10 ==0:
        print(f"epoch {epoch} :loss = {loss.data}")

epoch 0 :loss = 5.28415584564209
epoch 10 :loss = 0.17859184741973877
epoch 20 :loss = 0.0060446239076554775
epoch 30 :loss = 0.0002046627050731331
epoch 40 :loss = 6.9305206125136465e-06
epoch 50 :loss = 2.347103276179041e-07
epoch 60 :loss = 7.936043822098782e-09
epoch 70 :loss = 2.687984834714996e-10
epoch 80 :loss = 8.396412037869894e-12
epoch 90 :loss = 3.506528292555877e-13


In [12]:
print("learned weight:", model.weight.data)
print("learned bias:", model.bias.data)


learned weight: [[1.9999996]]
learned bias: [0.9999998]


In [13]:
import gzip
import struct
import urllib.request
from pathlib import Path

import numpy as np


In [14]:
DATA_DIR = PROJECT_ROOT / "data/mnist"
DATA_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://storage.googleapis.com/cvdf-datasets/mnist/"

FILES = {
    "train_images": "train-images-idx3-ubyte.gz",
    "train_labels": "train-labels-idx1-ubyte.gz",
    "test_images": "t10k-images-idx3-ubyte.gz",
    "test_labels": "t10k-labels-idx1-ubyte.gz",
}

for filename in FILES.values():
    path = DATA_DIR / filename
    if not path.exists():
        urllib.request.urlretrieve(BASE_URL + filename, path)
        print("downloaded", filename)


In [15]:
def load_mnist_images(path):
    with gzip.open(path, "rb") as f:
        magic, num, rows, cols = struct.unpack(">IIII", f.read(16))
        images = np.frombuffer(f.read(), dtype=np.uint8)
        images = images.reshape(num, rows, cols)
    return images

def load_mnist_labels(path):
    with gzip.open(path, "rb") as f:
        magic, num = struct.unpack(">II", f.read(8))
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    return labels


In [16]:
X_train = load_mnist_images(DATA_DIR / FILES["train_images"])
y_train = load_mnist_labels(DATA_DIR / FILES["train_labels"])

X_test = load_mnist_images(DATA_DIR / FILES["test_images"])
y_test = load_mnist_labels(DATA_DIR / FILES["test_labels"])

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


(60000, 28, 28) (60000,)
(10000, 28, 28) (10000,)


In [17]:
X_train = X_train.reshape(-1, 784).astype(np.float32) / 255.0
X_test = X_test.reshape(-1, 784).astype(np.float32) / 255.0

print(X_train.shape, X_test.shape)


(60000, 784) (10000, 784)


In [18]:
import numpy as np
from raptor import Tensor
from raptor import nn
from raptor.optim import Adam


In [19]:
model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)

criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-3)


In [20]:
batch_size = 64
epochs = 5

for epoch in range(epochs):
    perm = np.random.permutation(len(X_train))
    X_train_shuffled = X_train[perm]
    y_train_shuffled = y_train[perm]

    epoch_loss = 0.0
    correct = 0
    total = 0

    for i in range(0, len(X_train_shuffled), batch_size):
        X_batch = X_train_shuffled[i:i + batch_size]
        y_batch = y_train_shuffled[i:i + batch_size]

        x = Tensor(X_batch, requires_grad=False)
        logits = model(x)
        loss = criterion(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += float(loss.data) * len(X_batch)

        preds = np.argmax(logits.data, axis=1)
        correct += np.sum(preds == y_batch)
        total += len(X_batch)

    train_loss = epoch_loss / total
    train_acc = correct / total

    test_correct = 0
    test_total = 0

    for i in range(0, len(X_test), batch_size):
        X_batch = X_test[i:i + batch_size]
        y_batch = y_test[i:i + batch_size]

        x = Tensor(X_batch, requires_grad=False)
        logits = model(x)

        preds = np.argmax(logits.data, axis=1)
        test_correct += np.sum(preds == y_batch)
        test_total += len(X_batch)

    test_acc = test_correct / test_total

    print(
        f"epoch {epoch + 1}: "
        f"train_loss={train_loss:.4f}, "
        f"train_acc={train_acc:.4f}, "
        f"test_acc={test_acc:.4f}"
    )


epoch 1: train_loss=0.2790, train_acc=0.9185, test_acc=0.9595
epoch 2: train_loss=0.1144, train_acc=0.9656, test_acc=0.9703
epoch 3: train_loss=0.0787, train_acc=0.9758, test_acc=0.9729
epoch 4: train_loss=0.0579, train_acc=0.9823, test_acc=0.9712
epoch 5: train_loss=0.0470, train_acc=0.9852, test_acc=0.9760


In [21]:
from raptor.utils import batch_iterator, accuracy_from_logits, evaluate_classifier

for epoch in range(epochs):
    epoch_loss = 0.0
    total = 0
    correct = 0

    for X_batch, y_batch in batch_iterator(X_train, y_train, batch_size=64, shuffle=True):
        x = Tensor(X_batch, requires_grad=False)
        logits = model(x)
        loss = criterion(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += float(loss.data) * len(X_batch)
        correct += np.sum(np.argmax(logits.data, axis=1) == y_batch)
        total += len(X_batch)

    train_loss = epoch_loss / total
    train_acc = correct / total
    test_acc = evaluate_classifier(model, X_test, y_test, batch_size=64)

    print(
        f"epoch {epoch+1}: "
        f"train_loss={train_loss:.4f}, "
        f"train_acc={train_acc:.4f}, "
        f"test_acc={test_acc:.4f}"
    )


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 10 is different from 64)